# Phase 2 — Turn gameplay into training examples

**Goal:** turn 22 recorded sessions (388k frames) into ~97k training examples, each one:

- **input:** 16 grayscale 84×84 frames, every 4th frame
- **label:** the joystick action at the middle frame (index 8), as one of 9 classes

and split them into **train / val / test by whole session**.

Three ideas to understand before running anything:

**1. Store frames once, build clips on the fly.** Neighbouring clips overlap: clip *k+1* is clip *k* shifted by 4 raw frames, so they share 15 of 16 frames. The original repo saved every clip separately, which stores each frame ~16 times (~11 GB). We save each frame **once** (~2.7 GB) plus a list of clip start positions. When training asks for clip *k*, we grab `frames[start + 0, 4, 8, …, 60]`. Same data, 16× less disk.

**2. Split by session, not by clip.** If clip *k* is in train and clip *k+1* is in test, the test clip is a near-copy of a training clip. The model gets graded on things it has basically already seen, and the score looks better than it really is. That's called **data leakage**, and the original repo had it. We put whole sessions in train, val or test, so test is genuinely new gameplay.

**3. Labels come from `src/labels.py`.** One shared, tested mapping from action names. No hard-coded numbers anywhere.

**Time:** step 4 reads ~388k images and takes roughly 15–40 minutes on a laptop. It saves progress per session, so if you stop it, re-running picks up where it left off.

## 1. Setup and paths

This finds your project folder (the one with `README.md`) so paths work no matter where the notebook runs from. It also finds the Phase 1 download, whether that landed in `data/raw` or `notebooks/data/raw`.

In [ ]:
import os, sys, tarfile, io, json
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# --- find the project root (the folder that has README.md) ---
ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "README.md")) and os.path.dirname(ROOT) != ROOT:
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
print("Project root:", ROOT)

# --- find where Phase 1 put the unzipped data ---
candidates = [os.path.join(ROOT, "data", "raw", "ms_pacman"),
              os.path.join(ROOT, "notebooks", "data", "raw", "ms_pacman")]
RAW_DIR = next((p for p in candidates if os.path.isdir(p)), None)
assert RAW_DIR, f"Couldn't find the unzipped data in any of: {candidates}"
print("Raw data:    ", RAW_DIR)

OUT_DIR = os.path.join(ROOT, "data", "processed")
TRIAL_DIR = os.path.join(OUT_DIR, "trials")      # one file per session (lets us resume)
os.makedirs(TRIAL_DIR, exist_ok=True)
print("Output:      ", OUT_DIR)

SEQ, SKIP, IMG = 16, 4, 84
SPAN = (SEQ - 1) * SKIP + 1     # 61 raw frames per clip
MID = (SEQ // 2) * SKIP         # the label sits 32 raw frames after the clip starts

## 2. The label mapping (from `src/labels.py`)

This is the moment the notebook *imports* code instead of defining it. Open `src/labels.py` and read it. It's short.

The table below shows where every raw ID ends up. Check it against the `action_enums.txt` printout from Phase 1.

In [ ]:
from src.labels import ALE_ACTIONS, CLASS_NAMES, H_NAMES, V_NAMES, to_hv, to_9

for raw, name in enumerate(ALE_ACTIONS):
    h, v = to_hv(raw)
    print(f"{raw:2d} {name:14s} -> class {to_9(raw)} {CLASS_NAMES[to_9(raw)]:9s} (h={H_NAMES[h]}, v={V_NAMES[v]})")

**Why a 3×3 grid?** Class index = `3*v + h`, so the 9 classes line up like a joystick:

```
UPLEFT    UP     UPRIGHT        0 1 2
LEFT      NOOP   RIGHT    =     3 4 5
DOWNLEFT  DOWN   DOWNRIGHT      6 7 8
```

That makes the Phase 6 experiment (separate horizontal and vertical heads) trivial: `h = class % 3`, `v = class // 3`.

## 3. Find the sessions

Same search as Phase 1.

In [ ]:
trials = []
for root, _, fnames in os.walk(RAW_DIR):
    for fn in fnames:
        if fn.endswith(".tar.bz2"):
            stem = fn[:-len(".tar.bz2")]
            if os.path.exists(os.path.join(root, stem + ".txt")):
                trials.append((root, stem))
trials.sort(key=lambda t: t[1])
print(f"{len(trials)} sessions")

## 4. Process each session → frames + labels

For each session:

1. Read the label file: `frame_id`, `episode_id` and `action` for every row.
2. Read the PNGs **straight out of the `.tar.bz2`** without unzipping them to disk (saves ~5 GB), convert to 84×84 grayscale.
3. Match images to labels **by name** and keep them in label-file order.
4. Save `frames` (uint8), `actions` and `episodes` for that session.

**Why track episodes?** One session can contain several games: you die, a new game starts. A clip that spans a game-over shows a sudden jump from one maze to another, which isn't a real action. We'll skip those clips in step 5.

In [ ]:
def read_label_rows(txt_path):
    rows = []
    with open(txt_path) as f:
        f.readline()                                   # header
        for line in f:
            p = line.strip().split(",")
            if len(p) > 5:
                try:
                    rows.append((p[0], p[1], int(p[5])))   # frame_id, episode_id, action
                except ValueError:
                    pass                               # 'null' action -> skip the row
    return rows


def process_trial(root, stem):
    out_path = os.path.join(TRIAL_DIR, stem + ".npz")
    if os.path.exists(out_path):
        return "skipped (already done)"

    rows = read_label_rows(os.path.join(root, stem + ".txt"))
    wanted = {fid for fid, _, _ in rows}

    images = {}
    with tarfile.open(os.path.join(root, stem + ".tar.bz2"), "r:bz2") as tar:
        for member in tar:                             # streams through the archive once
            name = os.path.basename(member.name)
            if not name.endswith(".png") or name[:-4] not in wanted:
                continue
            img = Image.open(io.BytesIO(tar.extractfile(member).read()))
            images[name[:-4]] = np.asarray(img.convert("L").resize((IMG, IMG), Image.BILINEAR), dtype=np.uint8)

    kept = [(fid, ep, a) for fid, ep, a in rows if fid in images]
    frames = np.stack([images[fid] for fid, _, _ in kept])
    actions = np.array([a for _, _, a in kept], dtype=np.int16)
    episodes = np.array([ep for _, ep, _ in kept])
    np.savez(out_path, frames=frames, actions=actions, episodes=episodes)
    return f"{len(kept):,} frames kept of {len(rows):,} labeled"


for root, stem in tqdm(trials, desc="sessions"):
    print(stem, "->", process_trial(root, stem))

**Check:** "kept" should be close to "labeled" for every session. A big gap means images and labels aren't matching by name. Tell me if you see one.

## 5. Combine into one frame array + build the clip index

Now we stack all sessions into one big `frames.npy` and list every valid clip. A clip is valid when all 61 raw frames it spans are in the **same session and the same game (episode)**.

In [ ]:
trial_files = [os.path.join(TRIAL_DIR, stem + ".npz") for _, stem in trials]
sizes = [len(np.load(p)["actions"]) for p in trial_files]
total = sum(sizes)
print(f"{total:,} frames total -> {total * IMG * IMG / 1e9:.2f} GB on disk")

# open_memmap writes straight to disk, so this never needs 2.7 GB of RAM
frames_all = np.lib.format.open_memmap(os.path.join(OUT_DIR, "frames.npy"), mode="w+",
                                       dtype=np.uint8, shape=(total, IMG, IMG))
actions_all = np.zeros(total, dtype=np.int16)

starts, trial_of_clip = [], []
offset = 0
for t, p in enumerate(tqdm(trial_files, desc="combining")):
    d = np.load(p)
    n = len(d["actions"])
    frames_all[offset:offset + n] = d["frames"]
    actions_all[offset:offset + n] = d["actions"]
    eps = d["episodes"]
    for s in range(0, n - SPAN + 1, SKIP):             # a new clip every 4 raw frames
        if eps[s] == eps[s + SPAN - 1]:                # whole clip inside one game
            starts.append(offset + s)
            trial_of_clip.append(t)
    offset += n

frames_all.flush()
starts = np.array(starts, dtype=np.int64)
trial_of_clip = np.array(trial_of_clip, dtype=np.int16)
labels9 = np.array([to_9(a) for a in actions_all[starts + MID]], dtype=np.int64)
print(f"{len(starts):,} clips")

## 6. Split by session

We shuffle the **sessions** (with a fixed seed so you get the same split every time) and assign whole sessions: about 70% of sessions to train, 15% to val, 15% to test.

- **train:** what the model learns from
- **val:** what we check during training to pick the best version and tune settings
- **test:** touched **once**, at the very end, for the number you report. If you peek at test while tuning, it stops being an honest score.

In [ ]:
rng = np.random.default_rng(42)
order = rng.permutation(len(trials))
n_val = n_test = max(1, round(0.15 * len(trials)))
split_of_trial = {}
for i, t in enumerate(order):
    split_of_trial[t] = "test" if i < n_test else "val" if i < n_test + n_val else "train"

splits = {}
for name in ["train", "val", "test"]:
    mask = np.array([split_of_trial[t] == name for t in trial_of_clip])
    splits[name] = {"starts": starts[mask], "labels": labels9[mask]}
    sess = sorted(trials[t][1] for t, s in split_of_trial.items() if s == name)
    print(f"{name:5s}: {len(sess):2d} sessions, {mask.sum():>7,} clips")

np.savez(os.path.join(OUT_DIR, "splits.npz"),
         **{f"{k}_{f}": v[f] for k, v in splits.items() for f in ("starts", "labels")})
json.dump({"seq": SEQ, "skip": SKIP, "img": IMG, "mid": MID, "class_names": CLASS_NAMES,
           "sessions": {s: [trials[t][1] for t in range(len(trials)) if split_of_trial[t] == s]
                        for s in ["train", "val", "test"]}},
          open(os.path.join(OUT_DIR, "meta.json"), "w"), indent=2)
print("saved splits.npz and meta.json")

## 7. Check the class balance in each split

Each split should have roughly the same mix as the full dataset from Phase 1. If one split had zero DOWNLEFT clips, for example, we couldn't measure DOWNLEFT there.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
w = 0.27
for k, name in enumerate(["train", "val", "test"]):
    c = np.bincount(splits[name]["labels"], minlength=9)
    ax.bar(np.arange(9) + (k - 1) * w, 100 * c / c.sum(), w, label=f"{name} ({c.sum():,} clips)")
ax.set_xticks(range(9)); ax.set_xticklabels(CLASS_NAMES, rotation=30)
ax.set_ylabel("% of clips"); ax.set_title("Class mix per split"); ax.legend()
plt.tight_layout(); plt.show()

for name in ["train", "val", "test"]:
    c = np.bincount(splits[name]["labels"], minlength=9)
    print(f"{name:5s}", "  ".join(f"{CLASS_NAMES[i]}:{c[i]}" for i in range(9)))

## 8. Read a clip back the way training will

This uses **only the saved files**, exactly how the training code in Phase 4 will. If the picture looks like the Phase 1 clips, the pipeline works end to end.

In [ ]:
frames = np.load(os.path.join(OUT_DIR, "frames.npy"), mmap_mode="r")   # memory-mapped: reads from disk on demand
sp = np.load(os.path.join(OUT_DIR, "splits.npz"))
OFFSETS = np.arange(SEQ) * SKIP                                        # [0, 4, 8, ..., 60]

def get_clip(split, k):
    s = sp[f"{split}_starts"][k]
    return frames[s + OFFSETS], sp[f"{split}_labels"][k]               # (16, 84, 84) uint8, class index

k = len(sp["train_starts"]) // 2                                    # a clip from the middle of train
clip, y = get_clip("train", k)
print("clip shape:", clip.shape, clip.dtype, "| label:", CLASS_NAMES[y])

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for i, ax in enumerate(axes.flat):
    ax.imshow(clip[i], cmap="gray"); ax.axis("off")
    ax.set_title(("LABEL " if i == SEQ // 2 else "") + f"#{i}", color="red" if i == SEQ // 2 else "black", fontsize=9)
fig.suptitle(f"train clip {k}: {CLASS_NAMES[y]}", fontsize=13)
plt.tight_layout(); plt.show()

## Done with Phase 2

You now have, in `data/processed/`:

| File | What it is |
| --- | --- |
| `frames.npy` | every frame once, (N, 84, 84) uint8 |
| `splits.npz` | clip start positions + labels for train / val / test |
| `meta.json` | settings, class names, which sessions went where |
| `trials/` | per-session working files. Safe to delete once `frames.npy` exists |

`get_clip` in step 8 is the seed of the PyTorch `Dataset` in Phase 3.

**For `notes.md`:** how many clips per split, the class mix, how long step 4 took, and why splitting by session matters (one line in your own words; that line is basically a paragraph of the article).